# DeepJSCC vs NTSCC — two-environment integration

This notebook orchestrates repository scripts; it does not copy model implementations. DeepJSCC/digital evaluation runs in the Windows environment and NTSCC runs in its WSL Python 3.8 environment. Results meet through versioned JSON files on the shared `D:` drive.

**Invariant:** `CBR = complex channel uses / (3×H×W)`. Continuous JSCC symbols are not called bits. NTSCC entropy BPP is reported separately from radio use.

In [ ]:
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

SEMCOM_ROOT = Path(os.environ.get(
    "SEMCOM_ROOT",
    r"D:\OneDrive - Amrita vishwa vidyapeetham\Desktop\image_semcom",
))
REPO_ROOT = SEMCOM_ROOT / "SemanticSchedulerEnd2End"
RESULTS_DIR = REPO_ROOT / "results" / "runs"
IMAGE_PATH = SEMCOM_ROOT / "Deep-JSCC-PyTorch" / "demo" / "kodim08.png"
DEEP_CHECKPOINT = (
    SEMCOM_ROOT / "Deep-JSCC-PyTorch" / "out" /
    "imagenet_10_0.33_200.00_32_19.pth"
)
NTSCC_JSON = RESULTS_DIR / "ntscc_kodim08_10db.json"
SNR_DB = 10.0
TRIALS = 5

required = [REPO_ROOT / "src", IMAGE_PATH, DEEP_CHECKPOINT]
missing = [str(path) for path in required if not path.exists()]
print("Python:", sys.executable)
print("Repository:", REPO_ROOT)
print("Missing Windows inputs:", missing or "none")

## Run NTSCC in WSL

The official NTSCC source already defines `cbr_y` as **complex** channel uses divided by `3×H×W`. Do not divide it by two again. The exporter also records `bpp_y+bpp_z` as an entropy diagnostic and the untransmitted rate-map index cost separately.

In [ ]:
# Set True only from Windows when WSL and the ntscc38 environment are available.
RUN_NTSCC = False

wsl_repo = "/mnt/d/OneDrive - Amrita vishwa vidyapeetham/Desktop/image_semcom/SemanticSchedulerEnd2End"
wsl_results = f"{wsl_repo}/results/runs"
ntscc_command = (
    f'cd "{wsl_repo}" && '
    f'python scripts/run_ntscc_benchmark.py '
    f'--output-dir "{wsl_results}" --snr-db {SNR_DB:g} '
    f'--trials {TRIALS} --device cpu'
)
print("WSL command:\n", ntscc_command)
if RUN_NTSCC:
    completed = subprocess.run(
        ["wsl.exe", "bash", "-lc", ntscc_command],
        text=True, capture_output=True, check=True,
    )
    print(completed.stdout)
else:
    print("Not executed. Run the printed command in the NTSCC WSL environment.")

## Run DeepJSCC and the actual digital baselines

DeepJSCC maps `[0,1]` RGB directly to a continuous latent. The canonical repository AWGN layer pairs real values into per-image I/Q symbols, normalizes power and adds noise using Es/N0. The digital rows use real 5G-LDPC/QPSK/AWGN and report PSNR/SSIM only for CRC-valid frames.

In [ ]:
# Run this cell in the Windows environment after the NTSCC JSON exists.
RUN_WINDOWS_BENCHMARK = False

command = [
    sys.executable,
    str(REPO_ROOT / "scripts" / "run_communication_benchmark.py"),
    "--image", str(IMAGE_PATH),
    "--snr-db", str(SNR_DB),
    "--trials", str(TRIALS),
    "--output-dir", str(RESULTS_DIR),
]
if NTSCC_JSON.exists():
    command += ["--ntscc-result", str(NTSCC_JSON)]
print("Windows command:\n", subprocess.list2cmdline(command))
if RUN_WINDOWS_BENCHMARK:
    subprocess.run(command, cwd=REPO_ROOT, check=True)
else:
    print("Not executed. Set RUN_WINDOWS_BENCHMARK=True in the correct environment.")

## Validate and display the comparison

This cell refuses internally inconsistent CBR values. A missing model is shown as missing; no hard-coded PSNR/SSIM fallback is inserted.

In [ ]:
comparison_json = RESULTS_DIR / f"communication_comparison_{SNR_DB:g}db.json"
if not comparison_json.exists():
    raise FileNotFoundError(
        f"Run the benchmark first; missing {comparison_json}"
    )

rows = json.loads(comparison_json.read_text(encoding="utf-8"))
width, height = Image.open(IMAGE_PATH).size
for row in rows:
    expected = row["Complex_channel_uses"] / (3 * height * width)
    if not math.isclose(row["CBR"], expected, rel_tol=1e-9, abs_tol=1e-12):
        raise ValueError(f"Inconsistent CBR in row: {row['Method']}")

columns = [
    "Method", "SNR_dB", "Complex_channel_uses", "CBR",
    "Frame_success_rate", "PSNR_dB_mean", "SSIM_mean", "Qualification",
]
comparison = pd.DataFrame(rows)
display(comparison[[column for column in columns if column in comparison]])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_rows = comparison.dropna(subset=["PSNR_dB_mean"])
axes[0].scatter(plot_rows["CBR"], plot_rows["PSNR_dB_mean"])
for _, row in plot_rows.iterrows():
    axes[0].annotate(row["Method"], (row["CBR"], row["PSNR_dB_mean"]), fontsize=8)
axes[0].set(xlabel="CBR (complex uses / RGB value)", ylabel="PSNR (dB)", title="Rate–distortion")
axes[0].grid(alpha=0.25)
axes[1].barh(comparison["Method"], comparison["Complex_channel_uses"])
axes[1].set(xlabel="Complex channel uses", title="Radio resource")
axes[1].set_xscale("log")
plt.tight_layout()

## Inspect latents, parameters and accounting

DeepJSCC's `latent_real_values` is a continuous tensor length; NTSCC's `y` shape is an analysis-transform representation. Neither FP32 storage size is transmitted bits. Parameter counts describe model size, not per-image rate.

In [ ]:
def load_result(prefix: str):
    candidates = sorted(RESULTS_DIR.glob(f"{prefix}_*_{SNR_DB:g}db.json"))
    return json.loads(candidates[-1].read_text(encoding="utf-8")) if candidates else None

deep = load_result("deepjscc")
ntscc = load_result("ntscc")
inspection = []
if deep:
    inspection.append({
        "Model": "DeepJSCC",
        "Latent": f"{deep['latent_shape_per_tile']} × {deep['tile_count']} tiles",
        "Continuous real values": deep["latent_real_values"],
        "Complex uses": deep["complex_channel_uses"],
        "CBR": deep["cbr"],
        "Entropy-estimated bits": None,
        "Parameters": deep["parameters"]["total_parameters"],
    })
if ntscc:
    inspection.append({
        "Model": "NTSCC",
        "Latent": ntscc["y_latent_shape"],
        "Continuous real values": None,
        "Complex uses": ntscc["complex_channel_uses"],
        "CBR": ntscc["cbr"],
        "Entropy-estimated bits": ntscc["entropy_estimated_bits"],
        "Parameters": ntscc["parameters"]["total_parameters"],
    })
display(pd.DataFrame(inspection))

## K=256 codebook integration

For latent tokens shaped `B×N×C`, a fixed K=256 codebook uses 8 index bits per token. Per-image packet cost is `8N + rate_mask_bits + metadata_bits + CRC_bits`. Shared codebook storage (`256×C×precision`) is counted once, not per image. Send the real packet through the repository LDPC/QPSK path; a CRC failure remains a failed frame. Evaluate noiseless quantization separately from over-air performance.

For NTSCC, use a codebook per rate class (or product/residual VQ), because allocated dimension varies by patch, and transmit the rate-map indexes. Fine-tune with the quantizer in the loop; a post-hoc codebook normally adds distortion.

In [ ]:
def k256_packet_accounting(
    token_count: int,
    vector_dimension: int,
    rate_mask_bits: int = 0,
    metadata_bits: int = 128,
    crc_bits: int = 32,
    codebook_precision_bits: int = 16,
):
    index_bits = 8
    per_image_bits = token_count * index_bits + rate_mask_bits + metadata_bits + crc_bits
    shared_codebook_bits = 256 * vector_dimension * codebook_precision_bits
    return {
        "K": 256,
        "index_bits": index_bits,
        "per_image_packet_bits": per_image_bits,
        "shared_codebook_bits_once": shared_codebook_bits,
    }

# Example for one DeepJSCC 19×29×29 latent: 29×29 tokens of dimension 19.
display(pd.DataFrame([k256_packet_accounting(29 * 29, 19)]))

## Interpretation

A valid table tells you the quality/resource trade-off and channel robustness. DeepJSCC is fixed dense analog JSCC; NTSCC is content-adaptive nonlinear-transform JSCC; JPEG/LDPC is a separation baseline with CRC-visible cliff behavior. One image is only a smoke test. Final claims need the full Kodak set, several random channel trials, an SNR sweep, matched-CBR operating points, confidence intervals, latency/memory, all side information, and downstream task accuracy if the claim is semantic.